# Lab 5: Remote Job Market Pipeline

**Week 5 · Data Engineering Course**

---

## What You Will Build

An end-to-end pipeline that scrapes a live job board and lands it in a database:

```
Job board  ──Playwright──>  JSONL (raw)  ──pandas──>  CSV + Parquet  ──psycopg2──>  PostgreSQL
                                                                                        │
                                                                                        └─> SQL answers
```

Concretely:

1. **Scrape** all 100 postings from the Fake Python job board with Playwright, then follow 25 of them to their detail pages for the full description
2. **Store the raw output untouched** as JSON Lines, so every later step can be re-run without re-scraping
3. **Validate** the harvest — and discover something about the dataset that changes how you can use it
4. **Clean and derive** the columns a database table needs
5. **Save** a Parquet file for the pipeline and a CSV for humans
6. **Load** into a PostgreSQL table with an idempotent upsert
7. **Answer five questions** about the job market in SQL

Everything you built in Lessons 1–4 appears here, in the order a real pipeline runs it.

---

## Prerequisites

Lessons 1–4 complete, plus:

```
pip install playwright requests beautifulsoup4 pandas pyarrow psycopg2-binary python-dotenv
playwright install chromium
```

PostgreSQL running, with a database for this week:

```sql
CREATE DATABASE jobs_db;
```

And a `.env` file in the **`week5/`** folder (not in `labs/`):

```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=jobs_db
DB_USER=postgres
DB_PASSWORD=your_password_here
```

> **`.env` must never be committed.** Check that `.gitignore` lists it before your first commit this week.

---

## The target: `realpython.github.io/fake-jobs/`

A job board built for scraping practice. 100 postings, each with a detail page, and it does not change — so your selectors will still work next month.

**An honest note on tooling.** This board is server-rendered, so `requests` + BeautifulSoup would scrape it perfectly well and faster. We use Playwright anyway, for three reasons: you need the practice before you meet a board that *requires* it; the listing-then-detail flow is identical on a JavaScript-driven site; and the exercises ask you to point this same script at a JS-rendered page, where it will be the only thing that works. In a real project, you would check View Page Source first and use the cheapest tool that does the job — as Lesson 3 argued.

---

## Step 1: Setup

In [ ]:
import os
import json
import logging
import subprocess
import sys
import pandas as pd
import psycopg2
from pathlib import Path
from datetime import datetime, timezone

try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')         # the .env lives in week5/, we are in week5/labs/
except ImportError:
    print('python-dotenv not installed — falling back to real environment variables')

DATA  = Path('..') / 'data'
RAW   = DATA / 'raw'
CLEAN = DATA / 'clean'
LOGS  = Path('..') / 'logs'
for p in [RAW, CLEAN, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOGS / 'lab5.log', encoding='utf-8'),
    ],
    force=True,            # replace any handlers a previous run left behind
)
log = logging.getLogger('lab5')

log.info('Lab 5 started')
print('Setup complete.')

---

## Step 2: Write the Scraper as a Script

The scraper goes in a **`.py` file**, not in notebook cells. Three reasons, and they are the same three reasons every production scraper is a script:

1. **It can be scheduled.** In Week 7 you will hand a file like this to Airflow. You cannot schedule a notebook cell.
2. **It uses the sync Playwright API**, which is simpler to read and immune to the event-loop problems a notebook can have — especially on Windows (see Lesson 3.2).
3. **It separates fetching from analysis.** The script's only job is to put raw data on disk. Everything after that reads the file. When your cleaning logic turns out to be wrong at step 5, you fix it and re-run in one second instead of re-scraping for two minutes.

Note what the script does and does not do:

- it does **not** clean anything — raw strings go to disk exactly as scraped, plus a `scraped_at` timestamp
- it writes **JSONL**, one record per line, so a crash halfway through still leaves you everything up to that point
- it **logs** what it is doing, to the same log file as this notebook
- it takes **command-line arguments**, so you can run a 5-row test before a 100-row run
- it **screenshots the page if the expected selector never appears** — your only evidence when a scheduled run returns nothing

`%%writefile` is a Jupyter magic: it writes the rest of the cell to a file instead of executing it.

In [ ]:
%%writefile scrape_jobs.py
"""Scrape the Fake Python job board with Playwright.

Writes one JSON object per line to ../data/raw/jobs_raw.jsonl.
Nothing is cleaned here: this script's only job is to get the raw
data onto disk so the rest of the pipeline never needs the network.

Usage:
    python scrape_jobs.py                  # listing only (100 postings)
    python scrape_jobs.py --details 25     # also fetch 25 detail pages
    python scrape_jobs.py --limit 5        # a quick smoke test
    python scrape_jobs.py --headed         # watch the browser work
"""
import argparse
import json
import logging
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright
from playwright.sync_api import TimeoutError as PlaywrightTimeout

BOARD = 'https://realpython.github.io/fake-jobs/'
HERE  = Path(__file__).resolve().parent
DATA  = HERE.parent / 'data'
LOGS  = HERE.parent / 'logs'
SHOTS = DATA / 'screenshots'
USER_AGENT = 'DE-Course-Scraper/1.0 (student project; contact: you@example.com)'

log = logging.getLogger('scrape_jobs')


def setup_logging():
    LOGS.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s  %(levelname)-8s  %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[logging.StreamHandler(sys.stdout),
                  logging.FileHandler(LOGS / 'lab5.log', encoding='utf-8')],
    )


# ---------------------------------------------------------------- parsing
# These functions take HTML strings and return rows. No browser, no network
# -- which means you can test them against a saved .html file.

def parse_listing(html, page_url):
    """Parse the job cards on the listing page."""
    soup = BeautifulSoup(html, 'html.parser')
    rows = []

    for card in soup.select('div.card-content'):
        def text_at(selector):
            found = card.select_one(selector)
            return found.get_text(strip=True) if found else None

        # The 'Apply' link in the footer is the one that points at the detail page
        job_url = None
        for a in card.select('footer a'):
            if a.get_text(strip=True).lower() == 'apply' and a.get('href'):
                job_url = urljoin(page_url, a['href'])

        posted = card.select_one('time')

        rows.append({
            'title':      text_at('h2.title'),
            'company':    text_at('h3.company'),
            'location':   text_at('p.location'),
            'posted':     posted.get('datetime') if posted else None,
            'job_url':    job_url,
            'scraped_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        })

    return rows


def parse_detail(html):
    """Parse the fields that only exist on a posting's own page."""
    soup = BeautifulSoup(html, 'html.parser')

    # The description is the only <p> in .content without an id
    # (the others are the #location and #date paragraphs)
    description = None
    for p in soup.select('.content p'):
        if not p.get('id'):
            description = p.get_text(strip=True)
            break

    def after_label(selector):
        found = soup.select_one(selector)
        if not found:
            return None
        # '<strong>Location:</strong> Stewartbury, AA' -> 'Stewartbury, AA'
        return found.get_text(strip=True).split(':', 1)[-1].strip()

    return {
        'description':    description,
        'detail_location': after_label('#location'),
        'detail_posted':   after_label('#date'),
    }


# ---------------------------------------------------------------- scraping

def save_evidence(page, name):
    """Screenshot and dump the HTML of a page that did not look as expected."""
    SHOTS.mkdir(parents=True, exist_ok=True)
    page.screenshot(path=str(SHOTS / f'{name}.png'), full_page=True)
    (SHOTS / f'{name}.html').write_text(page.content(), encoding='utf-8')
    log.error(f'Saved evidence to {SHOTS / name}.png / .html')


def scrape(details=0, limit=None, headed=False, pause=0.5):
    """Scrape the board. Returns a list of raw row dicts."""
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=not headed)
        context = browser.new_context(user_agent=USER_AGENT,
                                      viewport={'width': 1280, 'height': 900})
        page = context.new_page()

        # --- listing page ---
        log.info(f'Opening {BOARD}')
        page.goto(BOARD, wait_until='domcontentloaded')
        try:
            page.wait_for_selector('div.card-content', timeout=15_000)
        except PlaywrightTimeout:
            save_evidence(page, 'listing_failed')
            browser.close()
            raise RuntimeError('No job cards appeared — the board may have changed')

        rows = parse_listing(page.content(), page.url)
        log.info(f'Parsed {len(rows)} postings from the listing page')

        if limit:
            rows = rows[:limit]
            log.info(f'--limit {limit}: keeping {len(rows)} rows')

        # --- detail pages ---
        n_details = min(details, len(rows))
        for i, row in enumerate(rows[:n_details], start=1):
            if not row['job_url']:
                continue
            try:
                page.goto(row['job_url'], wait_until='domcontentloaded')
                page.wait_for_selector('div.content', timeout=10_000)
                row.update(parse_detail(page.content()))
                log.info(f'  detail {i}/{n_details}: {row["title"]}')
            except PlaywrightTimeout:
                log.warning(f'  detail {i}/{n_details}: timed out on {row["job_url"]}')
                save_evidence(page, f'detail_failed_{i}')
            time.sleep(pause)          # be polite between requests

        browser.close()

    return rows


def write_jsonl(rows, path):
    """One JSON object per line. Appendable, streamable, crash-tolerant."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    log.info(f'Wrote {len(rows)} rows to {path}')


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--details', type=int, default=0,
                    help='how many postings to also fetch detail pages for')
    ap.add_argument('--limit', type=int, default=None,
                    help='keep only the first N postings (for testing)')
    ap.add_argument('--headed', action='store_true',
                    help='show the browser window')
    ap.add_argument('--out', default=str(DATA / 'raw' / 'jobs_raw.jsonl'))
    args = ap.parse_args()

    setup_logging()
    log.info(f'Scrape starting (details={args.details}, limit={args.limit})')

    rows = scrape(details=args.details, limit=args.limit, headed=args.headed)
    write_jsonl(rows, Path(args.out))

    log.info('Scrape complete')
    print(f'\n{len(rows)} postings -> {args.out}')


if __name__ == '__main__':
    main()

---

## Step 3: Run the Scraper

Smoke-test it first. `--limit 3 --details 3` touches four pages and finishes in seconds; if something is wrong with your Playwright install or the board's HTML, you find out now rather than two minutes into the full run.

Then run it for real. The full run is 100 listing rows plus 25 detail pages at half a second each — roughly 30 seconds.

In [ ]:
# Smoke test: 3 postings, 3 detail pages
!python scrape_jobs.py --limit 3 --details 3 --out ../data/raw/smoke_test.jsonl

In [ ]:
# The real run. If this cell fails, read ../logs/lab5.log and check
# ../data/screenshots/ for the evidence the scraper saved.
!python scrape_jobs.py --details 25

---

## Step 4: Load the Raw Output

From here on, nothing touches the network. Every cell below reads the file the scraper wrote, which means you can re-run this entire section as many times as you like for free.

In [ ]:
raw_path = RAW / 'jobs_raw.jsonl'
raw = pd.read_json(raw_path, lines=True)

log.info(f'Loaded {len(raw)} raw rows from {raw_path}')
print(f'{raw.shape[0]} rows x {raw.shape[1]} columns')
print(f'file size: {raw_path.stat().st_size / 1024:.0f} KB')
print()
print(raw.dtypes)
raw.head(3)

---

## Step 5: Validate Before You Trust

Never go straight from ingestion to loading. Interrogate the harvest first:

- Did you get the number of rows you expected?
- Is any required field missing?
- Are there duplicates on what should be a unique key?
- Do the distributions make sense?

That last question is the one people skip, and it is the one that catches the interesting problems. Run the next two cells and look carefully at the `posted` column.

In [ ]:
print('Expected 100 postings, got:', len(raw))
print()
print('Missing values per column:')
print(raw.isna().sum())
print()
print('Duplicate job_url values:', raw['job_url'].duplicated().sum())
print('Duplicate (title, company) pairs:', raw.duplicated(subset=['title', 'company']).sum())
print()
print('Rows enriched with a description:', raw['description'].notna().sum())

In [ ]:
print('Distribution of `posted`:')
print(raw['posted'].value_counts())
print()
print('Distribution of the region suffix in `location`:')
print(raw['location'].str.split(',').str[-1].str.strip().value_counts())
print()
print('Companies with more than one posting:')
dupes = raw['company'].value_counts()
print(dupes[dupes > 1] if (dupes > 1).any() else '  none')

### What you just found

**Every posting has the same `posted` date.** All 100. That is not a bug in your scraper — it is a property of this sandbox, which generated every posting on the same day.

It matters, because it rules out a whole class of analysis. "Postings per month", "is hiring accelerating", "how stale is this listing" — none of those questions can be answered from this dataset, and a chart of them would be a lie told with real data.

So keep the column (it is what the source said) and **write the limitation down**. In a real project this goes in the dataset's documentation, and it is exactly the kind of thing a stakeholder needs to hear from you *before* they ask for a trend chart.

Two more findings worth noting:

- **Locations are fictional**: three invented region codes (`AA`, `AE`, `AP`). Fine to group by, meaningless as geography.
- **One company appears twice**, so `company` is not a key. `job_url` is — and that is what we will put the `UNIQUE` constraint on.

> This step is the habit that separates a data engineer from a script. The pipeline would have run perfectly happily without it, and shipped a dataset nobody had checked.

---

## Step 6: Clean and Derive

Now the Week 3 work: fix the types, split the compound fields, and derive the columns the analysis will want. Every transformation here is reproducible from the raw file.

The derived columns:

| Column | How |
|--------|-----|
| `city`, `region` | split `location` on the comma |
| `posted_date` | parse to a real `date` |
| `seniority` | keyword search on the title: senior / junior / mid |
| `role_family` | which discipline the title belongs to (engineering, management, ...) |
| `mentions_python` | does the description mention Python (only meaningful where we have one) |
| `scraped_at` | when the row was collected — **always record this** |

`scraped_at` is not optional. Scraped data is a snapshot of a moment, and six months from now the only way to know whether a row is stale is the timestamp you stored with it.

In [ ]:
jobs = raw.copy()

# --- 1. Trim every string column -----------------------------------------
str_cols = ['title', 'company', 'location', 'description']
for col in str_cols:
    if col in jobs:
        jobs[col] = jobs[col].astype('string').str.strip()

# --- 2. Split the compound location field --------------------------------
parts = jobs['location'].str.split(',', n=1, expand=True)
jobs['city']   = parts[0].str.strip()
jobs['region'] = parts[1].str.strip() if parts.shape[1] > 1 else None

# --- 3. Real dates -------------------------------------------------------
jobs['posted_date'] = pd.to_datetime(jobs['posted'], errors='coerce').dt.date
jobs['scraped_at']  = pd.to_datetime(jobs['scraped_at'], errors='coerce', utc=True)

# --- 4. Seniority from the title ----------------------------------------
title_lower = jobs['title'].str.lower()
jobs['seniority'] = 'unspecified'
jobs.loc[title_lower.str.contains('senior|lead|head|principal|chief', na=False), 'seniority'] = 'senior'
jobs.loc[title_lower.str.contains('junior|graduate|trainee|intern', na=False),   'seniority'] = 'junior'

# --- 5. Role family from the title --------------------------------------
# Dict order is PRIORITY order: the first pattern to match wins, so
# 'Data scientist' lands in `data` rather than `science`. Without that rule
# the result would depend on which pattern happened to be checked last.
ROLE_PATTERNS = {
    'data':         r'\bdata\b|analyst|statistic',
    'engineering':  r'engineer|developer|programmer|architect|software',
    'health':       r'nurse|doctor|therap|psycholog|medical|pharmac|optic|dentist',
    'education':    r'teacher|lecturer|tutor|education|librarian',
    'creative':     r'designer|writer|editor|artist|producer|photographer|journalist',
    'science':      r'scientist|research|chemist|physicist|biolog|geolog',
    'management':   r'manager|director|officer|executive|administrator|consultant',
}
jobs['role_family'] = 'other'
for family, pattern in ROLE_PATTERNS.items():
    unclassified = jobs['role_family'].eq('other')
    jobs.loc[unclassified & title_lower.str.contains(pattern, na=False, regex=True), 'role_family'] = family

# --- 6. Skill flags from the description (where we have one) -------------
desc_lower = jobs['description'].str.lower()
jobs['mentions_python'] = desc_lower.str.contains('python', na=False)
jobs['mentions_sql']    = desc_lower.str.contains(r'\bsql\b', na=False, regex=True)

# --- 7. Deduplicate on the real key -------------------------------------
before = len(jobs)
jobs = jobs.drop_duplicates(subset=['job_url']).reset_index(drop=True)
log.info(f'Deduplicated on job_url: {before} -> {len(jobs)} rows')

# --- 8. Final column selection ------------------------------------------
jobs = jobs[['job_url', 'title', 'company', 'city', 'region', 'posted_date',
             'seniority', 'role_family', 'description',
             'mentions_python', 'mentions_sql', 'scraped_at']]

print(jobs.dtypes)
jobs.head()

In [ ]:
# Validate the cleaned frame. An assertion that fires here is a bug
# you catch before the database does.
assert len(jobs) > 0,                              'No rows survived cleaning'
assert jobs['job_url'].is_unique,                  'job_url is not unique'
assert jobs['job_url'].notna().all(),              'Some rows have no job_url'
assert jobs['title'].notna().all(),                'Some rows have no title'
assert jobs['company'].notna().all(),              'Some rows have no company'
assert jobs['posted_date'].notna().all(),          'Some rows have no posted_date'
assert jobs['seniority'].isin(['senior', 'junior', 'unspecified']).all(), 'Bad seniority value'

log.info(f'Validation passed: {len(jobs)} clean rows')
print(f'Validation passed — {len(jobs)} clean rows\n')

print('role_family (a keyword heuristic — expect a large "other" bucket,')
print('and say so when you report it):')
print(jobs['role_family'].value_counts())
print()
print('seniority:')
print(jobs['seniority'].value_counts())
print()
print(f'rows with a description: {jobs["description"].notna().sum()}')

---

## Step 7: Save CSV and Parquet

Lesson 4's rule, applied: **Parquet to stay inside the pipeline, CSV to leave it.**

The Parquet file keeps the dates as dates and the booleans as booleans, so the next stage needs no `parse_dates`. The CSV is for a human to open — written with `utf-8-sig` so it survives Excel — and drops the long description column that makes a spreadsheet unreadable.

At 100 rows the size difference is nothing; the **types** are the whole point. Parquet's size advantage shows up two orders of magnitude later, and by then you want to have been writing Parquet all along.

In [ ]:
parquet_path = CLEAN / 'jobs.parquet'
csv_path     = CLEAN / 'jobs.csv'

# Pipeline copy: everything, types intact
jobs.to_parquet(parquet_path, index=False)

# Human copy: no description, Excel-safe encoding
(jobs.drop(columns=['description'])
     .to_csv(csv_path, index=False, encoding='utf-8-sig'))

log.info(f'Saved {parquet_path} and {csv_path}')

print(f'{str(parquet_path):<34} {parquet_path.stat().st_size / 1024:>7.1f} KB  (all 12 columns)')
print(f'{str(csv_path):<34} {csv_path.stat().st_size / 1024:>7.1f} KB  (11 columns, no description)')
print()
print('At 100 rows the two sizes are about the same — Lesson 4 said as much.')
print('The reason to use Parquet here is not size, it is that posted_date comes')
print('back a date and mentions_python comes back a boolean, with no parse_dates.')

# Prove the round-trip is lossless
check = pd.read_parquet(parquet_path)
print()
print(f'read back: {check.shape}')
print(f'posted_date type: {type(check.loc[0, "posted_date"]).__name__}')
print(f'mentions_python type: {check["mentions_python"].dtype}')

---

## Step 8: Load into PostgreSQL

The table design, and why each part is there:

```sql
job_url     TEXT PRIMARY KEY   -- the natural key: one posting, one URL
title       TEXT NOT NULL
company     TEXT NOT NULL
city        TEXT
region      TEXT
posted_date DATE
seniority   TEXT
role_family TEXT
description TEXT
mentions_python BOOLEAN
mentions_sql    BOOLEAN
scraped_at  TIMESTAMPTZ NOT NULL  -- when WE collected it
loaded_at   TIMESTAMPTZ DEFAULT now()
```

`job_url` is the **primary key** rather than a `SERIAL` id, because it is the one value the source guarantees is unique per posting. That is what makes the upsert possible: `ON CONFLICT (job_url) DO UPDATE` means re-running this pipeline tomorrow **refreshes** the 100 rows instead of adding 100 more.

`scraped_at` and `loaded_at` answer two different questions — when the data was true, and when it entered your warehouse. You will meet that distinction again in Week 6 as part of slowly changing dimensions.

In [ ]:
DB_CONFIG = {
    'host':     os.environ.get('DB_HOST', 'localhost'),
    'port':     int(os.environ.get('DB_PORT', 5432)),
    'dbname':   os.environ.get('DB_NAME', 'jobs_db'),
    'user':     os.environ.get('DB_USER', 'postgres'),
    'password': os.environ.get('DB_PASSWORD', ''),
}

print(f'Connecting to {DB_CONFIG["dbname"]} at {DB_CONFIG["host"]}:{DB_CONFIG["port"]} '
      f'as {DB_CONFIG["user"]}')

try:
    conn = psycopg2.connect(**DB_CONFIG)
    log.info('Connected to PostgreSQL')
    print(f'Connected. Server version: {conn.server_version}')
except psycopg2.OperationalError as e:
    log.error(f'Connection failed: {e}')
    print('\nCheck that PostgreSQL is running, that jobs_db exists,')
    print('and that ../.env has the right credentials.')
    raise

In [ ]:
create_sql = '''
    CREATE TABLE IF NOT EXISTS job_postings (
        job_url         TEXT        PRIMARY KEY,
        title           TEXT        NOT NULL,
        company         TEXT        NOT NULL,
        city            TEXT,
        region          TEXT,
        posted_date     DATE,
        seniority       TEXT,
        role_family     TEXT,
        description     TEXT,
        mentions_python BOOLEAN,
        mentions_sql    BOOLEAN,
        scraped_at      TIMESTAMPTZ NOT NULL,
        loaded_at       TIMESTAMPTZ NOT NULL DEFAULT now()
    );
'''

with conn.cursor() as cur:
    cur.execute(create_sql)
    # An index on the columns we group by. Week 2's lesson, applied.
    cur.execute('CREATE INDEX IF NOT EXISTS idx_job_postings_region ON job_postings (region);')
    cur.execute('CREATE INDEX IF NOT EXISTS idx_job_postings_company ON job_postings (company);')
conn.commit()

log.info('Table job_postings ready')
print('Table job_postings ready.')

with conn.cursor() as cur:
    cur.execute('''
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_name = 'job_postings'
        ORDER BY ordinal_position;
    ''')
    for col, dtype, nullable in cur.fetchall():
        print(f'  {col:<17} {dtype:<26} null={nullable}')

In [ ]:
COLUMNS = ['job_url', 'title', 'company', 'city', 'region', 'posted_date',
           'seniority', 'role_family', 'description',
           'mentions_python', 'mentions_sql', 'scraped_at']

upsert_sql = """
    INSERT INTO job_postings
        (job_url, title, company, city, region, posted_date,
         seniority, role_family, description,
         mentions_python, mentions_sql, scraped_at)
    VALUES
        (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (job_url) DO UPDATE SET
        title           = EXCLUDED.title,
        company         = EXCLUDED.company,
        city            = EXCLUDED.city,
        region          = EXCLUDED.region,
        posted_date     = EXCLUDED.posted_date,
        seniority       = EXCLUDED.seniority,
        role_family     = EXCLUDED.role_family,
        -- keep an existing description if this run did not fetch one
        description     = COALESCE(EXCLUDED.description, job_postings.description),
        mentions_python = EXCLUDED.mentions_python,
        mentions_sql    = EXCLUDED.mentions_sql,
        scraped_at      = EXCLUDED.scraped_at,
        loaded_at       = now();
"""

# pandas NA values are not valid psycopg2 parameters — convert them to None
to_load = jobs[COLUMNS].astype(object).where(jobs[COLUMNS].notna(), None)
rows = list(to_load.itertuples(index=False, name=None))

print(f'Loading {len(rows)} rows...')
print('first row:', rows[0][:4], '...')

with conn.cursor() as cur:
    cur.executemany(upsert_sql, rows)
conn.commit()

log.info(f'Upserted {len(rows)} rows into job_postings')
print(f'Upserted {len(rows)} rows.')

In [ ]:
# Verify, then prove the load is idempotent by running it again
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM job_postings;')
    first_count = cur.fetchone()[0]

print(f'Rows in job_postings after load 1: {first_count}')
assert first_count == len(jobs), f'Expected {len(jobs)} rows, found {first_count}'

with conn.cursor() as cur:
    cur.executemany(upsert_sql, rows)          # the exact same load, again
conn.commit()

with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM job_postings;')
    second_count = cur.fetchone()[0]

print(f'Rows in job_postings after load 2: {second_count}')
assert second_count == first_count, 'The upsert created duplicates!'

log.info('Idempotency check passed')
print('\nIdempotency check passed — re-running the pipeline updates rows instead of duplicating them.')
print('This is the single most important property of a scheduled pipeline.')

---

## Step 9: Answer the Business Questions

Five questions this dataset can honestly answer. Notice that none of them is about time — Step 5 ruled that out.

In [ ]:
# Q1: How are postings distributed across regions?
q1 = pd.read_sql("""
    SELECT
        region,
        COUNT(*)                                        AS postings,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM job_postings
    GROUP BY region
    ORDER BY postings DESC;
""", conn)

print('Q1: Postings by region')
print(q1.to_string(index=False))

In [ ]:
# Q2: Which disciplines is this board actually hiring for?
q2 = pd.read_sql("""
    SELECT
        role_family,
        COUNT(*)                       AS postings,
        COUNT(DISTINCT company)        AS companies
    FROM job_postings
    GROUP BY role_family
    ORDER BY postings DESC;
""", conn)

print('Q2: Postings by role family')
print(q2.to_string(index=False))

In [ ]:
# Q3: Which words dominate the job titles?
# regexp_split_to_table turns one title into one row per word -- the SQL
# equivalent of pandas' .explode(), and a genuinely useful PostgreSQL trick.
q3 = pd.read_sql(r"""
    WITH words AS (
        SELECT lower(
                   unnest(
                       -- drop punctuation first, or you count 'engineer,' and
                       -- 'engineer' as two different words
                       regexp_split_to_array(regexp_replace(title, '[^a-zA-Z ]', ' ', 'g'), '\s+')
                   )
               ) AS word
        FROM job_postings
    )
    SELECT word, COUNT(*) AS occurrences
    FROM words
    WHERE length(word) > 3
      AND word NOT IN ('and', 'with', 'this', 'that', 'other')
    GROUP BY word
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC, word
    LIMIT 15;
""", conn)

print('Q3: Most common words in job titles')
print(q3.to_string(index=False))

In [ ]:
# Q4: Does any company have more than one posting?
q4 = pd.read_sql("""
    SELECT
        company,
        COUNT(*)                    AS postings,
        STRING_AGG(title, ' | ' ORDER BY title) AS roles
    FROM job_postings
    GROUP BY company
    HAVING COUNT(*) > 1
    ORDER BY postings DESC, company;
""", conn)

print('Q4: Companies with more than one posting')
print(q4.to_string(index=False) if len(q4) else '  none — every company posted exactly once')

In [ ]:
# Q5: Among the postings we enriched with a description, how many mention
# Python or SQL? Note the denominator: only the rows that HAVE a description.
q5 = pd.read_sql("""
    SELECT
        COUNT(*)                                          AS with_description,
        COUNT(*) FILTER (WHERE mentions_python)           AS mention_python,
        COUNT(*) FILTER (WHERE mentions_sql)              AS mention_sql,
        ROUND(100.0 * COUNT(*) FILTER (WHERE mentions_python) / COUNT(*), 1) AS pct_python
    FROM job_postings
    WHERE description IS NOT NULL;
""", conn)

print('Q5: Skill mentions among enriched postings')
print(q5.to_string(index=False))
print()
print('Reporting this as "% of all jobs on the board" would be wrong:')
print('we only fetched detail pages for 25 of the 100 postings.')

In [ ]:
# A last look at the table, then close the connection
sample = pd.read_sql("""
    SELECT title, company, region, seniority, role_family, posted_date
    FROM job_postings
    ORDER BY company
    LIMIT 10;
""", conn)
print(sample.to_string(index=False))

conn.close()
log.info('Lab 5 complete')
print('\nConnection closed. Lab 5 complete — see ../logs/lab5.log for the full run log.')

---

## Checklist

- [ ] `scrape_jobs.py` exists and runs from the command line
- [ ] `../data/raw/jobs_raw.jsonl` has 100 lines
- [ ] 25 of those rows have a `description`
- [ ] `../data/clean/jobs.parquet` and `../data/clean/jobs.csv` both exist
- [ ] `job_postings` in PostgreSQL has 100 rows
- [ ] Running the load twice leaves 100 rows, not 200 (the idempotency assertion passed)
- [ ] All five questions (Q1–Q5) are answered in the notebook
- [ ] `../logs/lab5.log` shows the full run, from the scrape to the load
- [ ] `.env` is in `.gitignore` and was **not** committed
- [ ] The notebook runs top to bottom with no errors

---

## Stretch Goals

Pick one. Each is a real pipeline feature, not busywork.

1. **Scrape all 100 detail pages** (`--details 100`) and re-run from Step 4. How long does it take? What would you change if it were 10,000 pages instead?
2. **Add a `job_skills` table.** One row per `(job_url, skill)` by searching each description for a list of skills, with a foreign key to `job_postings`. Then answer "which region most wants Python?" with a JOIN — Week 2's normalisation applied to scraped data.
3. **Cache the detail pages.** Save each page's HTML under `data/raw_html/` and read from there when it exists (Lesson 2.6). Re-running `--details 100` should then make zero requests.
4. **Make the scraper incremental.** Query the existing `job_url` values first and skip the detail fetch for postings already in the database. This is what every production scraper does, and it is the idea behind Week 6's incremental loading.
5. **Point the script at a JavaScript-rendered board.** Change `BOARD` to `https://quotes.toscrape.com/js/` and the selectors to match, and confirm your Playwright script handles it where `requests` could not.
6. **Add a data-quality report.** Before the load, write a small function that returns row count, null counts per column, and duplicate count — then fail the pipeline if any required column is more than 5% null. This is the seed of what Week 10 covers properly.